# Dynamic Conditional Correlation Modelling 

*By Vlad Popa*

Squared returns act as a proxy for time-varying variance $\sigma^2_t$. By means of the ACF Test, it is revealed that the variance of returns in certain periods tend to cluster together rather than occur randomly. This phenomenon is known as **volatility clustering**, and plays a significant role in risk management, derivative pricing and algorithmic trading because it proves that market volatility is not entirely random. By means of **observation-driven models**, which estimate unobserved time-varying parameters using lagged dependent variables, volatility clustering can be captured.

This project investigates volatility clustering by assessing the time-varying, conditional correlation between several assets, driven by geopolitical choke points in 2026 using the DCC-GARCH(1,1) model.

## 1. Import Libraries and Load Data

In [1]:
# Import libraries
library(rugarch)
library(rmgarch)
library(purrr)

# Read .csv files
bdi <- read.csv("input/mod/Modified Baltic Dry Index Historical Data.csv")
oil <- read.csv("input/mod/Modified Crude Oil WTI Futures Historical Data.csv")
soxx <- read.csv("input/mod/Modified SOXX ETF Stock Price History.csv")
urth <- read.csv("input/mod/Modified URTH ETF Stock Price History.csv")
gold <- read.csv("input/mod/Modified XAU_USD Historical Data.csv")

Loading required package: parallel


Attaching package: 'purrr'


The following object is masked from 'package:rugarch':

    reduce




In [2]:
# Save returns of each asset to a separate dataframe
bdi_rets <- data.frame(Date = bdi$Date,  BDI = bdi$Return / 100)
oil_rets <- data.frame(Date = oil$Date,  OIL = oil$Return / 100)
soxx_rets <- data.frame(Date = soxx$Date, SOXX = soxx$Return / 100)
urth_rets <- data.frame(Date = urth$Date, URTH = urth$Return / 100)
gold_rets <- data.frame(Date = gold$Date, GOLD = gold$Return / 100)

# Save all dataframes to a list
rets_list <- list(bdi_rets, oil_rets, soxx_rets, urth_rets, gold_rets)

# Sequentially merge each dataframe on the 'Date' column
returns_scaled <- purrr::reduce(rets_list, merge, by = "Date")
rownames(returns_scaled) <- returns_scaled$Date
returns_scaled$Date <- NULL

## 2. DCC-GARCH Model

The **DCC-GARCH model** is an extended, multivariate variant of an observation-driven model. It computes the conditional covariance matrix $\Sigma_t$ using lagged dependent variables, while also modelling the time-varying conditional correlation between the assets based on their conditional volatility. Unlike other multivariate GARCH models, the DCC extension allows for cross-causality when computing covariances, a small dimensionality, and a positive definite covariance matrix $\Sigma_t$. We bring the model together in the two sections below.

### 2.a GARCH Modelling

The first step of modelling the DCC-GARCH involves estimating an univariate GARCH model for each asset. In other words, we extract the behaviour of each individual asset's conditional variance. We do so by estimating the Exponential GARCH(1,1) Model (eGARCH(1,1)) on each asset, which also captures any asymmetric leverage effects (using $\gamma_i$) and ensures that an asset's variance never goes negative (though $\ln{(.)}$), as to obtain: $$\ln{(\sigma^2_{i,t})}=\omega_{i}+\beta_{i}\ln{(\sigma^2_{i,t-1})}+\alpha_{i}|\frac{y_{i,t-1}}{\sigma_{i,t-1}}| + \gamma_i \frac{y_{i,t-1}}{\sigma_{i,t-1}} \text{ for each asset } i \text{ in } i = 1,2,...$$

In [3]:
# Function for modelling conditional volatilities using eGARCH
my_garch <- function(returns_scaled){
    # Specify our GARCH model: how do individual variances behave?
    uspec <- ugarchspec(
        mean.model = list(armaOrder = c(0,0), include.mean = FALSE),
        variance.model = list(model = "sGARCH"),
        distribution.model = "norm")

    # Replicate the univariate GARCH model across all assets
    mspec <- multispec(replicate(ncol(returns_scaled), uspec))       

    # Estimate all conditional volatilities using our defined GARCH model
    fitlist <- multifit(multispec = mspec, data = returns_scaled)
    sigmas <- sigma(fitlist)
    rownames(sigmas) <- rownames(returns_scaled)
    colnames(sigmas) <- colnames(returns_scaled)
    
    # Save specifications, fits and conditional volatilities together
    output = list(
        sigmas = sigmas,
        mspec = mspec,
        fitlist = fitlist
    )
    return (output)
}

# Run function and save conditional volatilites
ugarch_results <- my_garch(returns_scaled) 
sigmas <- ugarch_results[["sigmas"]]
mspec <- ugarch_results[["mspec"]]
fitlist <- ugarch_results[["fitlist"]]
head(ugarch_results[["sigmas"]])

,BDI,OIL,SOXX,URTH,GOLD
2026-01-02,0.02765502,0.04374308,0.02827338,0.009258129,0.02091431
2026-01-05,0.02759484,0.04051297,0.02899411,0.009264485,0.01860048
2026-01-06,0.02753492,0.03802480,0.02871115,0.009270830,0.02133191
2026-01-07,0.02747506,0.03631624,0.02898010,0.009277164,0.01952609
2026-01-08,0.02741566,0.03460564,0.02864478,0.009283488,0.01814090
2026-01-09,0.02735648,0.03484459,0.02841817,0.009289801,0.01672695


### 2.b DCC Modelling

In this next step, we utilize the univariate eGARCH specifications from the previous section as well as the fitted data for each asset. By means of the DCC framework, we use those outputs as inputs for modelling the conditional correlation matrix based on standardized returns. In mathematical notation, we compute: 
$$Q_t = (1 - \alpha - \beta)\bar{Q} + \alpha(v_{t-1}v_{t-1}') + \beta Q_{t-1}$$

Where $v_t = \frac{y_t}{\sigma_t}$ represents the vector of standardized returns of all assets, and $\bar{Q}$ is the unconditional covariance matrix of the standardized residuals over the entire sample period. 

To ensure the elements are strictly bounded between $-1$ and $1$, the final conditional correlation matrix $R_t$ is rescaled as:

$$R_t = \text{diag}(Q_t)^{-1/2} Q_t \text{diag}(Q_t)^{-1/2}$$

In [6]:
# Function for modelling dynamic correlations between assets
my_dcc <- function(returns_scaled, mspec, fitlist){
    # Specify our DCC model: how do the correlations behave?
    dcc_spec <- dccspec(uspec = mspec, dccOrder = c(1,1), model = "DCC")
    dcc_fit <- dccfit(spec = dcc_spec, data = returns_scaled, fit = fitlist)
    corr_matrix <- rcor(dcc_fit)
    return (corr_matrix)
}

# Run function and save conditional correlation matrix
corr_matrix <- my_dcc(returns_scaled, mspec, fitlist) 